# AI-Powered Retail Sales Analytics and Anomaly Detection

**AICTE | IBM SkillsBuild Data Analytics with AI Academic Internship 2026**

### Project Objective
Analyze retail sales data, discover business patterns, visualize performance, and use machine learning to:
1. detect unusual sales transactions using **Isolation Forest**, and
2. predict sales using a **Random Forest Regressor**.

The notebook is designed to be reproducible and beginner-friendly.


## 1. Import Libraries

The project uses:
- **Pandas / NumPy** for data preparation and analysis
- **Matplotlib / Seaborn** for visualization
- **Scikit-learn** for AI/ML


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Load Dataset

Dataset source: **Sample Superstore**.

The CSV is loaded directly from a public GitHub dataset mirror so the notebook can run without manually downloading a file.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
df.head()


## 3. Data Understanding and Cleaning

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().copy()

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Order Date", "Sales", "Quantity", "Profit"])
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.strftime("%b")
df["Profit_Margin"] = np.where(df["Sales"] != 0, (df["Profit"] / df["Sales"]) * 100, 0)

print("Cleaned shape:", df.shape)
df.head()


## 4. Exploratory Data Analysis

We calculate key business KPIs and examine category, region, and time-based performance.


In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_quantity = df["Quantity"].sum()
avg_order_value = total_sales / total_orders

print(f"Total Sales       : ${total_sales:,.2f}")
print(f"Total Profit      : ${total_profit:,.2f}")
print(f"Total Orders      : {total_orders:,}")
print(f"Total Quantity    : {total_quantity:,}")
print(f"Average Order     : ${avg_order_value:,.2f}")


### Category Performance

In [ ]:
category_summary = (
    df.groupby("Category")[["Sales", "Profit", "Quantity"]]
      .sum()
      .sort_values("Sales", ascending=False)
)
display(category_summary)

plt.figure(figsize=(9,5))
sns.barplot(data=category_summary.reset_index(), x="Category", y="Sales")
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


### Region Performance

In [ ]:
region_summary = (
    df.groupby("Region")[["Sales", "Profit"]]
      .sum()
      .sort_values("Sales", ascending=False)
)
display(region_summary)

plt.figure(figsize=(9,5))
sns.barplot(data=region_summary.reset_index(), x="Region", y="Sales")
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


### Monthly Sales Trend

In [ ]:
monthly_sales = (
    df.set_index("Order Date")
      .resample("MS")["Sales"]
      .sum()
)

plt.figure(figsize=(12,5))
monthly_sales.plot()
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


## 5. Profitability Analysis

High sales do not always mean high profit. This section identifies categories and sub-categories with weak profitability.


In [ ]:
subcategory_profit = (
    df.groupby("Sub-Category")[["Sales", "Profit"]]
      .sum()
      .sort_values("Profit")
)

display(subcategory_profit)

plt.figure(figsize=(11,7))
sns.barplot(
    data=subcategory_profit.reset_index(),
    x="Profit",
    y="Sub-Category"
)
plt.title("Profit by Sub-Category")
plt.xlabel("Profit")
plt.ylabel("Sub-Category")
plt.tight_layout()
plt.show()


## 6. AI Model 1 — Sales Anomaly Detection

**Isolation Forest** is an unsupervised machine-learning algorithm used to identify transactions that behave differently from normal transactions.

Features used:
- Sales
- Quantity
- Discount
- Profit


In [ ]:
features = df[["Sales", "Quantity", "Discount", "Profit"]].copy()

iso_model = IsolationForest(
    contamination=0.02,
    random_state=42
)

df["Anomaly_Flag"] = iso_model.fit_predict(features)
df["Anomaly"] = np.where(df["Anomaly_Flag"] == -1, "Anomaly", "Normal")

anomaly_df = df[df["Anomaly"] == "Anomaly"].copy()

print("Detected anomalies:", len(anomaly_df))
display(
    anomaly_df[
        ["Order ID", "Order Date", "Category", "Sales", "Quantity", "Discount", "Profit", "Anomaly"]
    ].sort_values("Sales", ascending=False).head(15)
)


### Visualize Detected Anomalies

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df,
    x="Sales",
    y="Profit",
    hue="Anomaly",
    alpha=0.7
)
plt.title("AI-Based Sales Anomaly Detection")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()


## 7. AI Model 2 — Sales Prediction

A **Random Forest Regressor** is trained to estimate transaction-level sales using numerical and encoded categorical features.

This is a demonstration model for the internship project; it is not intended as a production forecasting system.


In [ ]:
model_df = df[[
    "Sales", "Quantity", "Discount", "Profit",
    "Category", "Sub-Category", "Region", "Segment"
]].copy()

model_df = pd.get_dummies(
    model_df,
    columns=["Category", "Sub-Category", "Region", "Segment"],
    drop_first=True
)

X = model_df.drop(columns=["Sales"])
y = model_df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=120,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
predictions = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R² Score: {r2:.4f}")


### Actual vs Predicted Sales

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, predictions, alpha=0.5)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.tight_layout()
plt.show()


## 8. AI-Generated Business Insight Summary

The following section converts model outputs and analytical results into concise business insights using programmatic logic. This makes the notebook runnable without an external AI API while still using machine learning for the AI component.


In [ ]:
best_category = category_summary["Sales"].idxmax()
best_region = region_summary["Sales"].idxmax()
most_profitable_subcategory = subcategory_profit["Profit"].idxmax()
least_profitable_subcategory = subcategory_profit["Profit"].idxmin()

print("AI-ASSISTED BUSINESS INSIGHTS")
print("=" * 60)
print(f"1. Highest sales category: {best_category}.")
print(f"2. Highest sales region: {best_region}.")
print(f"3. Most profitable sub-category: {most_profitable_subcategory}.")
print(f"4. Lowest-profit sub-category: {least_profitable_subcategory}.")
print(f"5. Isolation Forest detected {len(anomaly_df)} unusual transactions.")
print(f"6. Sales prediction model achieved R² = {r2:.3f} on the held-out test set.")

print("\nSuggested actions:")
print("- Investigate anomalous transactions before making operational decisions.")
print("- Compare high-sales categories with their profit contribution.")
print("- Review low-profit sub-categories for discount and pricing issues.")
print("- Use the prediction model as an analytical support tool, not as a final business decision maker.")


## 9. Conclusion

This project demonstrates an end-to-end data analytics workflow:
- data loading
- data cleaning
- exploratory analysis
- KPI calculation
- visualization
- AI-based anomaly detection
- machine-learning sales prediction
- business insight generation

### Key Learning Outcomes
The project combines traditional data analytics with machine learning so that raw retail transactions can be transformed into understandable business information.
